In [2]:
!nvidia-smi


Wed Oct 29 11:27:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.42                 Driver Version: 581.42         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4000             WDDM  |   00000000:2D:00.0  On |                  Off |
| 41%   41C    P0             37W /  140W |     726MiB /  16376MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

# Check PyTorch GPU availability
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Is Available: {torch.cuda.is_available()}")
else:
    print("GPU is not available.")

GPU Name: NVIDIA RTX A4000
GPU Is Available: True


In [10]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="IBNq5RIq2rtJ5gStlHrE")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(11)
dataset = version.download("yolov5")
                


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-11 in yolov5pytorch:: 100%|██████████| 20262/20262 [00:48<00:00, 419.49it/s]


In [11]:
from ultralytics import YOLO
import shutil, os
import torch

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\cl502_19\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [12]:

# Load YOLOv5 Model (nano/small/medium/large; use yolov5n.pt or yolov5s.pt for fast training)
model = YOLO("yolov5n.pt")

PRO TIP  Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



In [17]:
results = model.train(
    data= "C:/Users/cl502_19/Downloads/DL_ANPR/License-Plate-Recognition-11/data.yaml",
    epochs=100,              # More epochs for better accuracy (try 100–150)
    imgsz=640,               # Balanced image size
    batch=32,                # RTX A4000 (16 GB) can handle 32 at 640x640
    workers=4,               # Increase for faster data loading
    device=0,                # Use GPU 0
    optimizer="AdamW",       # Better convergence than SGD in many cases
    lr0=0.001,               # Starting learning rate
    lrf=0.01,                # Final learning rate
    patience=30,             # Early stop if no improvement for 30 epochs
    cos_lr=True,             # Cosine learning rate schedule (smoother)
    augment=True,            # Use built-in data augmentation
    cache="ram"              # Cache dataset in RAM for speed
)


Ultralytics 8.3.221  Python-3.10.8 torch-2.8.0+cu126 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:/Users/cl502_19/Downloads/DL_ANPR/License-Plate-Recognition-11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, 

In [21]:
import os
import shutil

# Path to your weights from the latest training (adjust train2 if your folder name is different)
src_best = r'C:\Users\cl502_19\Downloads\DL_ANPR\runs\detect\train2\weights\best.pt'
src_last = r'C:\Users\cl502_19\Downloads\DL_ANPR\runs\detect\train2\weights\last.pt'

# Destination folder in your project directory
dst_folder = r'C:\Users\cl502_19\Downloads\DL_ANPR\saved_models'
os.makedirs(dst_folder, exist_ok=True)

# Copy the weights
shutil.copy(src_best, os.path.join(dst_folder, 'license_plate_best.pt'))
shutil.copy(src_last, os.path.join(dst_folder, 'license_plate_last.pt'))
print(f"Weights saved in {dst_folder}/")


Weights saved in C:\Users\cl502_19\Downloads\DL_ANPR\saved_models/


# TEST


In [ ]:
from ultralytics import YOLO
import glob

model = YOLO("saved_models/license_plate_best.pt")
test_images = glob.glob("License-Plate-Recognition-11/test/images/*.jpg")





image 1/1 c:\Users\cl502_19\Downloads\DL_ANPR\License-Plate-Recognition-11\test\images\0002a5b67e5f0909_jpg.rf.c8f81ef986e3e99af6f349c200080453.jpg: 480x640 2 License_Plates, 23.1ms
Speed: 5.7ms preprocess, 23.1ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)
Results saved to C:\Users\cl502_19\Downloads\DL_ANPR\runs\detect\predict
Processed: License-Plate-Recognition-11/test/images\0002a5b67e5f0909_jpg.rf.c8f81ef986e3e99af6f349c200080453.jpg

image 1/1 c:\Users\cl502_19\Downloads\DL_ANPR\License-Plate-Recognition-11\test\images\000812dcf304a8e7_jpg.rf.ba32e6c184b3d974abcced6f7c29af6d.jpg: 576x640 2 License_Plates, 18.2ms
Speed: 9.1ms preprocess, 18.2ms inference, 3.5ms postprocess per image at shape (1, 3, 576, 640)
Results saved to C:\Users\cl502_19\Downloads\DL_ANPR\runs\detect\predict
Processed: License-Plate-Recognition-11/test/images\000812dcf304a8e7_jpg.rf.ba32e6c184b3d974abcced6f7c29af6d.jpg

image 1/1 c:\Users\cl502_19\Downloads\DL_ANPR\License-Plate-Recogni

In [36]:
from ultralytics import YOLO

model = YOLO("saved_models/license_plate_best.pt")
metrics = model.val(data="License-Plate-Recognition-11/data.yaml")  # path to your data.yaml (use abs/rel as appropriate)

# The results object contains various metrics:
print(metrics)


Ultralytics 8.3.221  Python-3.10.8 torch-2.8.0+cu126 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
YOLOv5n summary (fused): 84 layers, 2,503,139 parameters, 0 gradients, 7.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 46.516.2 MB/s, size: 21.5 KB)
val: Scanning C:\Users\cl502_19\Downloads\DL_ANPR\License-Plate-Recognition-11\valid\labels.cache... 2048 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2048/2048 949.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 6.3it/s 20.4s0.2s
                   all       2048       2195      0.985      0.948      0.979      0.717
Speed: 1.2ms preprocess, 2.5ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to C:\Users\cl502_19\Downloads\DL_ANPR\runs\detect\val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatri

In [38]:
print("mAP50:", metrics.box.map50)    # Mean Average Precision @ IoU=0.5
print("F1 score:", metrics.box.f1[0]) # F1 score for class 0 (license plate)
print("Precision:", metrics.box.p[0]) # Precision for class 0
print("Recall:", metrics.box.r[0])    # Recall for class 0


mAP50: 0.9786846605157783
F1 score: 0.9657319584770949
Precision: 0.9845624967088559
Recall: 0.9476082004555809


In [39]:
from ultralytics import YOLO

# Load your trained model
model = YOLO("saved_models/license_plate_best.pt")

# Path to your random image (adjust as needed)
random_img_path = "C:/Users/cl502_19/Downloads/DL_ANPR/image.jpeg"

# Run inference
results = model.predict(random_img_path, save=True, imgsz=640, conf=0.25)

print("Tested random image. Check runs/detect/predict/ for output!")



image 1/1 C:\Users\cl502_19\Downloads\DL_ANPR\image.jpeg: 640x384 2 License_Plates, 23.7ms
Speed: 4.0ms preprocess, 23.7ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 384)
Results saved to C:\Users\cl502_19\Downloads\DL_ANPR\runs\detect\predict2
Tested random image. Check runs/detect/predict/ for output!


In [ ]:
# end of model traning

In [7]:
# inddia dataset finetunig 

In [9]:
from ultralytics import YOLO

# Start from the previously trained best.pt
model = YOLO('C:/Users/cl502_19/Downloads/DL_ANPR/saved_models/license_plate_best.pt')

# Fine-tune on Indian dataset
results = model.train(
    data='C:/Users/cl502_19/Downloads/DL_ANPR/indian_dataset/data_1.yaml',
   epochs=100,              # More epochs for better accuracy (try 100–150)
    imgsz=640,               # Balanced image size
    batch=32,                # RTX A4000 (16 GB) can handle 32 at 640x640
    workers=4,               # Increase for faster data loading
    device=0,                # Use GPU 0
    optimizer="AdamW",       # Better convergence than SGD in many cases
    lr0=0.001,               # Starting learning rate
    lrf=0.01,                # Final learning rate
    patience=30,             # Early stop if no improvement for 30 epochs
    cos_lr=True,             # Cosine learning rate schedule (smoother)
    augment=True,            # Use built-in data augmentation
    cache="ram"              # Cache dataset in RAM for speed
)


New https://pypi.org/project/ultralytics/8.3.223 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.221  Python-3.10.8 torch-2.8.0+cu126 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:/Users/cl502_19/Downloads/DL_ANPR/indian_dataset/data_1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:/Users/cl502_19/Downloads/DL_ANPR/saved_models/license_plate_bes

In [2]:
import os
import shutil
import glob

# Try both possible patterns
patterns = ["runs/detect/train*/weights"]
exp_weights = []
for pattern in patterns:
    exp_weights.extend(glob.glob(pattern))

if not exp_weights:
    raise FileNotFoundError("No weights folder found. Check your training run and paths!")

latest_weights = sorted(exp_weights, key=os.path.getmtime)[-1]

best_pt = os.path.join(latest_weights, "best.pt")
last_pt = os.path.join(latest_weights, "last.pt")

dst_folder = r"C:\Users\cl502_19\Downloads\DL_ANPR\saved_models\finetune"
os.makedirs(dst_folder, exist_ok=True)

shutil.copy(best_pt, os.path.join(dst_folder, "indian_license_plate_best.pt"))
shutil.copy(last_pt, os.path.join(dst_folder, "indian_license_plate_last.pt"))
print(f"Fine-tuned weights saved as indian_license_plate_best.pt and indian_license_plate_last.pt in {dst_folder}")


Fine-tuned weights saved as indian_license_plate_best.pt and indian_license_plate_last.pt in C:\Users\cl502_19\Downloads\DL_ANPR\saved_models\finetune


In [1]:
# end